# שינוי מחירים — החודש האחרון מול החודש שלפניו

עובד מול `[cpi].[dbo].[foodChainPricesScd]`, כמו `cpi_scd.ipynb`.

1. זמני הטעינה נלקחים מ-`validfrom` / `validto`. החודש של הטעינה האחרונה הוא החודש הנוכחי (למשל נובמבר).
   - **אחרונה** (`last`) — כל הטעינות בחודש הנוכחי
   - **קודמת** (`prev`) — כל הטעינות בחודש שלפניו (למשל אוקטובר)
2. בכל טעינה לוקחים את המחיר שהיה בתוקף בכל סניף: `validfrom <= load_ts AND (validto > load_ts OR validto IS NULL)`.
3. לכל מוצר (`itemcode`) מחשבים ממוצע מחיר על פני כל הסניפים וכל הטעינות בכל חודש.
4. `change = avg_price_last / avg_price_prev - 1` — באחוזים, כמה המוצר התייקר בכל הארץ.

In [26]:
import pandas as pd
import pyodbc

SERVER = r'PCH80041449\MSSQLSERVER01'
DATABASE = 'cpi'
TABLE = '[cpi].[dbo].[foodChainPricesScd]'


def connect():
    return pyodbc.connect(driver='ODBC Driver 17 for SQL Server', server=SERVER,
                          database=DATABASE, trusted_connection='yes')

In [27]:
LOADS_SQL = '''
SELECT validfrom AS load_ts FROM {t}
UNION
SELECT validto FROM {t} WHERE validto IS NOT NULL
'''.format(t=TABLE)

con = connect()
all_loads = pd.read_sql(LOADS_SQL, con).load_ts
con.close()

anchor = all_loads.max()
cur_month = anchor.to_period('M')
prev_month = cur_month - 1
loads = all_loads[all_loads.dt.to_period('M') >= prev_month].sort_values().to_frame()
loads['month'] = loads.load_ts.dt.to_period('M').astype(str)
loads['period'] = 'prev'
loads.loc[loads.month == str(cur_month), 'period'] = 'last'
print('last = %s, prev = %s' % (cur_month, prev_month))
loads

last = 2026-11, prev = 2026-10


,load_ts,month,period
1,2026-10-07 10:44:48,2026-10,prev
2,2026-10-14 10:44:48,2026-10,prev
3,2026-10-21 10:44:48,2026-10,prev
4,2026-11-04 10:44:48,2026-11,last


In [28]:
PRICES_SQL = '''
SELECT v.itemcode, l.period, MAX(v.itemname) AS itemname,
       AVG(v.itemprice) AS avg_price,
       COUNT(DISTINCT v.chainid + '-' + v.subchainid + '-' + v.storeid) AS branches,
       COUNT_BIG(*) AS observations
FROM {t} v
JOIN #loads l ON l.load_ts >= v.validfrom AND (v.validto IS NULL OR l.load_ts < v.validto)
WHERE v.itemprice > 0 AND v.validfrom <= ? AND (v.validto IS NULL OR v.validto > ?)
GROUP BY v.itemcode, l.period
'''.format(t=TABLE)

con = connect()
cursor = con.cursor()
cursor.execute('CREATE TABLE #loads (load_ts datetime2(0) NOT NULL PRIMARY KEY, period varchar(4) NOT NULL)')
cursor.executemany('INSERT INTO #loads VALUES (?, ?)',
                   [(r.load_ts.to_pydatetime(), r.period) for r in loads.itertuples()])
prices = pd.read_sql(PRICES_SQL, con, params=[anchor.to_pydatetime(), loads.load_ts.min().to_pydatetime()])
con.close()
prices.groupby('period').itemcode.count()

period
last    5
prev    5
Name: itemcode, dtype: int64

In [29]:
names = prices.sort_values('period').groupby('itemcode').itemname.first()  # name from the last period when available
result = prices.pivot(index='itemcode', columns='period', values=['avg_price', 'branches'])
result.columns = ['%s_%s' % c for c in result.columns]
result = result.dropna(subset=['avg_price_last', 'avg_price_prev'])  # only items sold in both periods
result.insert(0, 'itemname', names)
result['change'] = result.avg_price_last / result.avg_price_prev - 1
result = result[['itemname', 'avg_price_prev', 'avg_price_last', 'change', 'branches_prev', 'branches_last']]
result.sort_values('change', ascending=False)

,itemname,avg_price_prev,avg_price_last,change,branches_prev,branches_last
itemcode,,,,,,
7290000000010,חלב 3% 1 ליטר,10.296296,10.777778,0.046763,9.0,9.0
7290000000034,ביצים L תריסר,31.000000,32.000000,0.032258,9.0,9.0
7290000000027,לחם אחיד פרוס 750 גרם,20.296296,20.888889,0.029197,9.0,9.0
7290000000058,שמן זית 750 מיליליטר,50.555556,51.666667,0.021978,9.0,9.0
7290000000041,קפה נמס 200 גרם,40.148148,40.888889,0.018450,9.0,9.0


In [30]:
result.change.describe()

count    5.000000
mean     0.029729
std      0.011001
min      0.018450
25%      0.021978
50%      0.029197
75%      0.032258
max      0.046763
Name: change, dtype: float64

## שינוי מחיר לכל מוצר בכל סניף — חודש A מול חודש B

הנתונים: דגימה שבועית (`DateInserted`) של כל סניף, מתוך `data/simulation/full_prices_simulation.csv`
(3 רשתות × 3 סניפים × 5 מוצרים, החל מ-2026-09-16 10:44:48).

1. הסניף מזוהה לפי `chainid` + `subchainid` + `storeid` (מתוך `filename`).
2. לכל סניף ומוצר: `avg_price` = ממוצע המחיר על פני כל הדגימות השבועיות בחודש (`samples` = מספר הדגימות).
3. `change = avg_price_B / avg_price_A - 1` — כמה הסניף העלה (או הוריד) את מחיר המוצר.
4. נכללים רק מוצרים שנדגמו בסניף בשני החודשים.

In [31]:
SIM_CSV = r'data\simulation\full_prices_simulation.csv'
MONTH_A = '2026-09'  # base month
MONTH_B = '2026-10'  # compared month

weekly = pd.read_csv(SIM_CSV, encoding='utf-8-sig', dtype={'itemcode': str}, parse_dates=['DateInserted'])
weekly = weekly[weekly.itemprice > 0]
weekly = weekly.join(weekly.filename.str.extract(
    r'^PriceFull(?P<chainid>\d+)-(?P<subchainid>\d+)-(?P<storeid>\d+)-'))
weekly['month'] = weekly.DateInserted.dt.to_period('M').astype(str)
weekly.groupby('month').DateInserted.agg(['min', 'max', 'nunique'])  # weekly samples per month

,min,max,nunique
month,,,
2026-09,2026-09-16 10:44:48,2026-09-30 10:44:48,3
2026-10,2026-10-07 10:44:48,2026-10-28 10:44:48,4
2026-11,2026-11-04 10:44:48,2026-11-11 10:44:48,2


In [32]:
BRANCH = ['chain', 'chainid', 'subchainid', 'storeid', 'itemcode']

monthly = (weekly[weekly.month.isin([MONTH_A, MONTH_B])]
           .groupby(BRANCH + ['month'])
           .itemprice.agg(['mean', 'size'])
           .rename(columns={'mean': 'avg_price', 'size': 'samples'})
           .unstack('month'))
monthly.columns = ['%s_%s' % c for c in monthly.columns]
price_a, price_b = 'avg_price_' + MONTH_A, 'avg_price_' + MONTH_B
samples_a, samples_b = 'samples_' + MONTH_A, 'samples_' + MONTH_B
branch_change = monthly.dropna(subset=[price_a, price_b]).reset_index()  # sold in the branch in both months
branch_change.insert(len(BRANCH), 'itemname',
                     branch_change.itemcode.map(weekly.groupby('itemcode').itemname.last()))
branch_change['change'] = branch_change[price_b] / branch_change[price_a] - 1
branch_change = branch_change[BRANCH + ['itemname', price_a, price_b, 'change', samples_a, samples_b]]
branch_change.style.format({price_a: '{:.2f}', price_b: '{:.2f}', 'change': '{:+.1%}',
                            samples_a: '{:.0f}', samples_b: '{:.0f}'})

,chain,chainid,subchainid,storeid,itemcode,itemname,avg_price_2026-09,avg_price_2026-10,change,samples_2026-09,samples_2026-10
0,Carrefour,7290055700007,001,001,7290000000010,חלב 3% 1 ליטר,10.00,10.00,+0.0%,3,4
1,Carrefour,7290055700007,001,001,7290000000027,לחם אחיד פרוס 750 גרם,20.00,20.00,+0.0%,3,4
2,Carrefour,7290055700007,001,001,7290000000034,ביצים L תריסר,30.00,30.00,+0.0%,3,4
3,Carrefour,7290055700007,001,001,7290000000041,קפה נמס 200 גרם,40.00,40.00,+0.0%,3,4
4,Carrefour,7290055700007,001,001,7290000000058,שמן זית 750 מיליליטר,50.00,50.00,+0.0%,3,4
5,Carrefour,7290055700007,001,002,7290000000010,חלב 3% 1 ליטר,10.00,11.00,+10.0%,3,4
6,Carrefour,7290055700007,001,002,7290000000027,לחם אחיד פרוס 750 גרם,20.00,20.00,+0.0%,3,4
7,Carrefour,7290055700007,001,002,7290000000034,ביצים L תריסר,30.00,30.00,+0.0%,3,4
8,Carrefour,7290055700007,001,002,7290000000041,קפה נמס 200 גרם,40.00,40.00,+0.0%,3,4
9,Carrefour,7290055700007,001,002,7290000000058,שמן זית 750 מיליליטר,50.00,50.00,+0.0%,3,4


In [33]:
# summary: branch x item
(branch_change.pivot_table(index=['chain', 'storeid'], columns='itemname', values='change')
 .style.format('{:+.1%}', na_rep='-'))